# Stage 1: Data Preparation

_Reformated by Claude for better readability_  
Pipeline: **load → clean HTML → vectorize → shuffle & partition**

## Step 1: Load Dataset

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress CUDA warnings
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import re
import numpy as np
from datasets import load_dataset
import keras


dataset = load_dataset("stanfordnlp/imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Step 2: Clean HTML

IMDB reviews contain `<br />`, `<b>`, etc. from web scraping — strip them with a single regex before they waste vocabulary slots.

In [20]:
print("=== Before cleanup ===")
for t in dataset["train"]["text"][19:22]:
    print(repr(t[:150]), "\n")

texts = [re.sub(r"<[^>]+>", " ", x) for x in dataset["train"]["text"]]

print("=== After cleanup ===")
for t in texts[19:22]:
    print(repr(t[:150]), "\n")

labels = np.array(dataset["train"]["label"])
pos = labels.mean()
print(f"Labels: {len(labels)} samples — {pos:.1%} positive, {1 - pos:.1%} negative")

=== Before cleanup ===
'Pros: Nothing<br /><br />Cons: Everything<br /><br />Plot summary: A female reporter runs into a hitchhiker that tells her stories about the deaths of' 

'If the crew behind "Zombie Chronicles" ever read this, here\'s some advice guys: <br /><br />1. In a "Twist Ending"-type movie, it\'s not a good idea to' 

'1st watched 8/3/2003 - 2 out of 10(Dir-Brad Sykes): Mindless 3-D movie about flesh-eating zombies in a 3 story within a movie chronicle. And yes, we g' 

=== After cleanup ===
'Pros: Nothing  Cons: Everything  Plot summary: A female reporter runs into a hitchhiker that tells her stories about the deaths of people that were ki' 

'If the crew behind "Zombie Chronicles" ever read this, here\'s some advice guys:   1. In a "Twist Ending"-type movie, it\'s not a good idea to insert cl' 

'1st watched 8/3/2003 - 2 out of 10(Dir-Brad Sykes): Mindless 3-D movie about flesh-eating zombies in a 3 story within a movie chronicle. And yes, we g' 

Labels: 25000 samples

## Step 3: Vectorize

Fit `TextVectorization` on the full training corpus so every client shares the same vocabulary mapping.

In [ ]:
vectorizer = keras.layers.TextVectorization(
    max_tokens=20_000, output_sequence_length=500, dtype="int32"
)
vectorizer.adapt(texts)

vocab = vectorizer.get_vocabulary()
print(f"Vocab size: {len(vocab)}")
print(f"Top 10 tokens: {vocab[:10]}")

Vocab size: 20000
Top 10 tokens: ['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it')]


## Step 4: Shuffle & Vectorize

Shuffle with a fixed seed so partitions are reproducible, then convert all texts to integer token sequences.

In [22]:
rng = np.random.default_rng(67)
idx = rng.permutation(25_000)

x = keras.ops.convert_to_numpy(vectorizer([texts[i] for i in idx]))
y = labels[idx]

print(f"x shape: {x.shape}  dtype: {x.dtype}")
print(f"y shape: {y.shape}  label balance: {y.mean():.1%} positive")
print(f"\nSample — first review, first 20 tokens:\n{x[0, :20]}")

x shape: (25000, 500)  dtype: int64
y shape: (25000,)  label balance: 50.0% positive

Sample — first review, first 20 tokens:
[  29   21   37   72   12 7805    1   67  110  960  830    2    1  416
 5579    7  176 1723    3   15]
